# 前馈网络（FFN）：Norm + GLU 家族手撕实现

本文件聚焦 **Pre-Norm 结构** 与 **GLU 激活家族**（与 `MLP.ipynb` 的 SwiGLU 互补）。

## 1. Pre-Norm
Transformer 每个子层：`x = x + Sublayer(Norm(x))`（先归一再进子层，Post-Norm 是先子层再归一）。Pre-Norm 训练更稳、可深层，LLaMA/GPT-Neo 等采用。

## 2. GLU 家族
$$\text{GLU}(x)=\sigma(W_1 x)\odot W_2 x,\quad \text{SwiGLU}(x)=\text{SiLU}(W_1 x)\odot W_2 x$$
门控：$\sigma(W_1 x)$ 决定 $W_2 x$ 各通道通过比例。SwiGLU 把 sigmoid 门换成 SiLU，平滑非单调，效果更好。最终再 $W_3$ 投影回原维。

## 3. hidden_dim 的 $2/3$
GLU 有 3 个矩阵，参数量约为标准 MLP 的 $3/2$，故 hidden_dim 乘 $2/3$ 对齐。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6) -> None:
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

class PreNorm(nn.Module):
    """先归一化再进子层：x -> fn(norm(x))"""
    def __init__(self, dim: int, fn: nn.Module, norm_type: str = 'rms') -> None:
        super().__init__()
        self.norm = RMSNorm(dim) if norm_type == 'rms' else nn.LayerNorm(dim)
        self.fn = fn
    def forward(self, x: torch.Tensor, **kw) -> torch.Tensor:
        return self.fn(self.norm(x), **kw)

class SwiGLU(nn.Module):
    """SwiGLU FFN：down(SiLU(gate(x)) * up(x))"""
    def __init__(self, dim: int, hidden_dim: int, bias: bool = False) -> None:
        super().__init__()
        self.gate = nn.Linear(dim, hidden_dim, bias=bias)   # W1
        self.up   = nn.Linear(dim, hidden_dim, bias=bias)   # W2
        self.down = nn.Linear(hidden_dim, dim, bias=bias)   # W3
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.down(F.silu(self.gate(x)) * self.up(x))

class LLaMAFeedForward(nn.Module):
    def __init__(self, dim: int, hidden_dim: int = None, multiple_of: int = 256, dropout: float = 0.0) -> None:
        super().__init__()
        if hidden_dim is None:
            hidden_dim = 4 * dim
        hidden_dim = int(2 * hidden_dim / 3)
        hidden_dim = multiple_of * ((hidden_dim + multiple_of - 1) // multiple_of)
        self.swiglu = SwiGLU(dim, hidden_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.swiglu(x))

In [ ]:
# 验证：PreNorm + FFN 组成一个 Transformer 子层
torch.manual_seed(0)
dim = 128
ffn = LLaMAFeedForward(dim)
sublayer = PreNorm(dim, ffn)               # PreNorm 包裹 FFN
x = torch.randn(2, 10, dim)
out = x + sublayer(x)                       # 残差连接：x + Sublayer(Norm(x))
print('子层输出 shape:', out.shape)
print('PreNorm 用 RMSNorm:', isinstance(sublayer.norm, RMSNorm))

## 小结 / 易错点
- **Pre-Norm** 是 `fn(norm(x))`，残差加在外面 `x + ...`；写反成 Post-Norm 会影响深网训练稳定性。
- GLU 的门控顺序：`act(W1 x) * W2 x` 再 `W3`，三个矩阵缺一不可。
- $2/3$ 缩放 + `multiple_of` 对齐是 LLaMA 的工程约定，非必须但常见。
- 原仓库把 RMSNorm、GLU、SwiGLU、sigmoid/softmax 混在一个文件，本版聚焦 FFN 相关，softmax/sigmoid 见 `激活函数.ipynb`。

In [ ]:
# ===== assert 测试验证 =====
torch.manual_seed(42)
dim = 64
ffn = LLaMAFeedForward(dim, hidden_dim=128)
x = torch.randn(2, 10, dim)
out = ffn(x)
assert out.shape == x.shape, f"FFN 输出形状错误: {out.shape}"
print(f"✅ LLaMAFeedForward: {x.shape} -> {out.shape}")

sublayer = PreNorm(dim, ffn)
out2 = x + sublayer(x)
assert out2.shape == x.shape
print(f"✅ PreNorm + 残差: {x.shape} -> {out2.shape}")

rms = RMSNorm(dim)
x3 = torch.randn(4, dim)
out3 = rms(x3)
assert out3.shape == x3.shape
norm_val = out3.pow(2).mean(dim=-1)
assert torch.allclose(norm_val, torch.ones_like(norm_val), atol=1e-3), "RMSNorm 后均方应为 1"
print(f"✅ RMSNorm: 均方≈1 (误差 {(norm_val-1).abs().max():.4f})")

sw = SwiGLU(dim, hidden_dim=128)
out4 = sw(x)
assert out4.shape == x.shape, f"SwiGLU 输出形状错误: {out4.shape}"
print(f"✅ SwiGLU: {x.shape} -> {out4.shape}")

ffn2 = LLaMAFeedForward(dim)
assert ffn2.swiglu.gate.weight.shape[0] % 256 == 0 or ffn2.swiglu.gate.weight.shape[0] == int(2*4*dim/3)
print(f"✅ hidden_dim 对齐: {ffn2.swiglu.gate.weight.shape[0]}")
print("✅ 全部测试通过")
